In [ ]:
"""
Converts the original Delile et al. (2019) data/object into the
AnnData (`adata.h5ad`) used by the notebooks in this repo.

Source: Delile, J. et al. (2019). Single cell transcriptomics reveals
spatial and temporal dynamics of gene expression in the developing mouse
spinal cord. Development, 146(12), dev173807.
Raw data: ArrayExpress E-MTAB-7320
Original analysis code: https://github.com/juliendelile/MouseSpinalCordAtlas

"""
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import scipy.sparse as sp

counts_path = "UMI_count.tsv"
pheno_path = "phenoData_annotated.csv"

pheno = pd.read_csv(pheno_path, sep="\t", index_col=0)

chunksize = 2000  
reader = pd.read_csv(counts_path, sep="\t", index_col=0, chunksize=chunksize)

gene_ids, chunks = [], []
for chunk in reader:
  gene_ids.extend(chunk.index.tolist())
  chunks.append(sp.csr_matrix(chunk.values))  # sparsify now, drop the dense chunk
del chunk

genes_by_cells = sp.vstack(chunks)
del chunks
counts = genes_by_cells.T.tocsr()  # transpose: AnnData wants cells x genes
del genes_by_cells

cell_ids = pd.read_csv(counts_path, sep="\t", nrows=0, index_col=0).columns.tolist()
adata = ad.AnnData(X=counts, obs=pheno.loc[cell_ids], var=pd.DataFrame(index=gene_ids))

neural_mask = adata.obs["Type_step1"].isin(["Neuron", "Progenitor"])
adata = adata[neural_mask].copy()

print(adata)
print(adata.obs["timepoint"].value_counts())



In [ ]:
import mygene

ens_genes = adata.var.index.to_list()

mg = mygene.MyGeneInfo()
results = mg.querymany(ens_genes, scopes="ensembl.gene", fields="symbol", species="mouse")

df = pd.DataFrame(results)
print(df[['query', 'symbol']])

In [ ]:
id_to_symbol = dict(zip(df['query'], df['symbol']))

adata.var['gene_symbol'] = adata.var_names.map(id_to_symbol)

adata.var['gene_symbol'] = adata.var['gene_symbol'].fillna(adata.var_names.to_series())

print(adata.var[['gene_symbol']].head())

adata.var_names = adata.var['gene_symbol']
adata.var_names_make_unique()

In [7]:
from scipy.stats import median_abs_deviation

def simple_qc(adata, mad_threshold = 3):
    
    adata.var["mt"] = adata.var_names.str.startswith(("MT-","mt-"))

    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt"],
        percent_top=None,
        log1p=False,
        inplace=True
    )

    n_genes_median = np.median(adata.obs["n_genes_by_counts"])
    n_genes_mad = median_abs_deviation(adata.obs["n_genes_by_counts"])

    lower_gene_limit = np.maximum(n_genes_median - (mad_threshold * n_genes_mad), 100)

    mt_median = np.median(adata.obs["pct_counts_mt"])
    mt_mad = median_abs_deviation(adata.obs["pct_counts_mt"])

    upper_mt_limit = np.minimum(mt_median + (mad_threshold * mt_mad), 20.0)
    
    adata = adata[
        (adata.obs["n_genes_by_counts"]>lower_gene_limit) &
        (adata.obs["pct_counts_mt"]<upper_mt_limit),:
    ]

    sc.pp.filter_genes(adata,min_cells=3)

    print(f"Simple QC Summary:")
    print(f" - Min Genes: {lower_gene_limit:.2f}")
    print(f" - Max MT %: {upper_mt_limit:.2f}")
    print(f" - Remaning Cells: {adata.n_obs}")

    return adata

In [ ]:
adata = simple_qc(adata)

In [9]:
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=2000,subset=True)

In [11]:
adata.write_h5ad("adata.h5ad")